https://cups.online/ru/training/8/tasks/1367

# Item-Based Collaborative Filtering with Popularity Boost

## Обозначения

Пусть:

* $U = {1,\dots,n}$ — множество пользователей
* $I = {1,\dots,m}$ — множество объектов
* $X \in {0,1}^{n \times m}$ — бинарная матрица взаимодействий

$$
X_{u,i} =

\begin{cases}
1, & \text{если пользователь} u \text{ взаимодействовал с объектом } i \
0, & \text{иначе}
\end{cases}
$$

---

## 1. L1-нормализация пользователей

Число взаимодействий пользователя:

$$
|I_u| = \sum_{j=1}^{m} X_{u,j}
$$

Строим нормализованную матрицу:

$$
\tilde{X}_{u,i} =
\begin{cases}
\dfrac{X_{u,i}}{|I_u|}, & \text{если} |I_u| > 0 \
0, & \text{иначе}
\end{cases}
$$

Тогда:

$$
\tilde{X} \in \mathbb{R}^{n \times m}
$$

---

## 2. Матрица ко-встречаемости объектов

$$
C = \tilde{X}^T X
$$

Покомпонентно:

$$
C_{i,j} = 
\sum_{u=1}^{n}
\frac{X_{u,i}}{|I_u|}
X_{u,j}
$$

Диагональ зануляется:

$$
C_{i,i} = 0
$$

---

## 3. Популярность объектов

Популярность:

$$
p_j = \sum_{u=1}^{n} X_{u,j}
$$

Параметр буста:

$$
\gamma > 0
$$

Диагональная матрица буста:

$$
D = \operatorname{diag}(p_1^\gamma, \dots, p_m^\gamma)
$$

---

## 4. Итоговая матрица весов

$$
W = C D
$$

Покомпонентно:

$$
W_{i,j} = C_{i,j} \cdot p_j^\gamma
$$

---

## 5. Скоринг пользователя

Для пользователя $u$:

$$
s_u = X_u W
$$

Покомпонентно:

$$
s_u(j) = 
\sum_{i \in I_u}
C_{i,j} \cdot p_j^\gamma
$$

---

## 6. Исключение просмотренных объектов

$$
s_u(j) = -\infty
\quad \text{если } X_{u,j} = 1
$$

---

## 7. Top-K рекомендации

$$
\mathcal{R}_{u} =
\operatorname{TopK}_{j \notin I_u} s_u(j)
$$

---

## 8. Полная формула модели

Подставляя $C = \tilde{X}^T X$, получаем:

$$
\hat{r}_{u,j}
=
\sum_{i \in I_u}
\left(
\sum_{v=1}^{n}
\frac{X_{v,i}}{|I_v|}
X_{v,j}
\right)
p_j^\gamma
$$

---

## Компактная матричная форма

$$
\boxed{
\hat{R} = X \tilde{X}^T X D
}
$$

где

* $X$ — бинарная user-item матрица
* $\tilde{X}$ — L1-нормализованная версия
* $D$ — диагональная матрица популярности


In [183]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
from scipy import sparse
from pathlib import Path

In [203]:
base_dir = Path('.')
train = pd.read_csv(base_dir / 'train.csv')
sample_sub = pd.read_csv(base_dir / 'sample_submission.csv', index_col=0)

In [185]:
train.head()

,user_id,course_id
0,39972,34
1,56815,51
2,63734,20
3,17896,81
4,36961,64


In [186]:
all_users = pd.concat([train.user_id, pd.Series(sample_sub.index)]).unique()
all_items = train.course_id.unique()

user_map = {uid: u for u, uid in enumerate(all_users)}
item_map = {iid: i for i, iid in enumerate(all_items)} 
inv_item_map = {v: k for k, v in item_map.items()}

train['u'] = train.user_id.map(user_map)
train['i'] = train.course_id.map(item_map)
n_users, n_items = len(all_users), len(all_items)

In [187]:
data = np.ones(len(train), dtype=np.float32)
X_binary = csr_matrix((data, (train['u'], train['i'])), shape=(n_users, n_items))
X_norm = normalize(X_binary, norm='l1', axis=1)
Cooc = X_norm.T.dot(X_binary)
Cooc.setdiag(0)

In [200]:
pop_items = np.array(X_binary.sum(axis=0)).flatten()
power = 12
boost_vec = np.power(pop_items, power)
boost_mat = sparse.diags(boost_vec)
W = Cooc.dot(boost_mat)

In [201]:
pd.DataFrame(W.toarray())

,0,1,2,3,4,5,6,7,8,9,...,161,162,163,164,165,166,167,168,169,170
0,0.000000e+00,1.098675e+30,4.464927e+29,2.843752e+14,0.000000e+00,1.601028e+33,3.640762e+25,2.286441e+28,8.288117e+25,1.049089e+26,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.083333,0.0,0.00
1,7.297053e+30,0.000000e+00,3.314347e+29,3.062502e+14,1.456130e+12,1.875242e+33,8.105640e+25,3.996696e+29,6.230307e+25,1.802541e+26,...,0.0,0.0,0.0,0.000000,0.0,512.0,0.0,0.083333,0.0,0.00
2,6.681705e+30,7.467802e+29,0.000000e+00,1.493903e+13,1.456130e+12,1.141776e+33,5.467997e+25,1.057926e+29,7.175260e+25,5.786619e+25,...,0.0,0.0,0.0,0.000000,197889312.0,1024.0,0.0,0.083333,0.0,0.00
3,6.064222e+29,9.832899e+28,2.128790e+27,0.000000e+00,2.588676e+12,3.325542e+32,1.852695e+25,1.598606e+28,0.000000e+00,7.175011e+23,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00
4,0.000000e+00,1.169156e+28,5.188926e+27,6.473581e+13,0.000000e+00,1.370313e+32,1.680352e+25,0.000000e+00,2.163595e+25,0.000000e+00,...,0.0,0.0,0.0,372.363647,0.0,0.0,0.0,0.000000,0.0,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,0.000000e+00,2.338311e+28,2.075571e+28,0.000000e+00,0.000000e+00,4.774395e+31,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00
167,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00
168,1.035355e+29,1.558874e+28,6.918569e+27,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00
169,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00


In [202]:
recs = []
global_pop = [inv_item_map[idx] for idx in np.argsort(pop_items)[::-1][:50]]
test_users = sample_sub.index.tolist()
batch_size = 500

for start in range(0, len(test_users), batch_size):
    end = min(start + batch_size, len(test_users))
    u_real = test_users[start:end]
    u_idx = [user_map[u] for u in u_real]

    user_hists = X_binary[u_idx]
    scores = user_hists.dot(W).toarray()
    
    rows, cols = user_hists.nonzero()
    scores[rows, cols] = -np.inf

    top_k = 5
    if scores.shape[1] < top_k: top_k = scores.shape[1]

    unsorted_idx  = np.argpartition(scores, -top_k, axis=1)[:, -top_k:]

    for i in range(len(u_real)):
        idx = unsorted_idx[i]
        idx = idx[np.argsort(scores[i, idx])[::-1]]

        rec = []
        for cand in idx:
            if scores[i, cand] > 0:
                rec.append(inv_item_map[cand])
            if len(rec) == 3: break
        
        if len(rec) < 3: 
            seen = set(user_hists[i].indices) 
            for gb in global_pop:
                if item_map[gb] not in seen and gb not in rec:
                    rec.append(gb)
                if len(rec) == 3: break    
        recs.append(rec)

pd.DataFrame(recs, columns=['course_id_1', 'course_id_2', 'course_id_3'], index=sample_sub.index).to_csv(base_dir / 'submission.csv')
        


In [197]:
recs = pd.DataFrame(recs, columns=['course_id_1', 'course_id_2', 'course_id_3'], index=sample_sub.index)

In [198]:
recs

,course_id_1,course_id_2,course_id_3
user_id,,,
78,1,7,15
81,15,7,3
120,34,15,7
123,7,1,15
150,1,7,3
...,...,...,...
185864,7,1,15
186262,7,1,15
186691,7,1,15


In [199]:
recs.to_csv(base_dir / 'submission.csv')